In [1]:
import pandas as pd
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score


def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_gold(gold_path):
    rows = []
    for obj in read_jsonl(gold_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "gold_rating": obj.get("rating"),
                "gold_comment": obj.get("comment"),
            }
        )
    return pd.DataFrame(rows)


def normalize_pred(pred_path):
    rows = []
    for obj in read_jsonl(pred_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "llm_rating": obj.get("rating"),
                "llm_comment": obj.get("comment"),
                "status": obj.get("status", "success"),
            }
        )
    df = pd.DataFrame(rows)
    if "status" in df.columns:
        df = df[df["status"] == "success"].copy()
    return df


def infer_dataset_name(folder_path):
    return Path(folder_path).name


def infer_model_name(pred_file_name):
    name = pred_file_name.replace(".jsonl", "")
    if name.startswith("gemini"):
        return "gemini"
    if name.startswith("qwen"):
        return "qwen"
    if name.startswith("claude"):
        return "claude"
    if name.startswith("llm"):
        return "llm"
    if name.startswith("gpt"):
        return "gpt"
    return name


def evaluate_one_pair(gold_path, pred_path):
    df_gold = normalize_gold(gold_path)
    df_pred = normalize_pred(pred_path)

    df = pd.merge(df_gold, df_pred, on="article_id", how="inner")

    total_merged = len(df)
    df = df.dropna(subset=["gold_rating", "llm_rating"]).copy()
    valid_rows = len(df)

    if valid_rows == 0:
        return None, None

    df["gold_rating"] = df["gold_rating"].astype(int)
    df["llm_rating"] = df["llm_rating"].astype(int)
    df["abs_error"] = (df["gold_rating"] - df["llm_rating"]).abs()

    labels = [-2, -1, 0, 1, 2]

    metrics = {
        "dataset": infer_dataset_name(Path(gold_path).parent),
        "model": infer_model_name(Path(pred_path).name),
        "gold_file": Path(gold_path).name,
        "pred_file": Path(pred_path).name,
        "gold_rows": len(df_gold),
        "pred_rows_success": len(df_pred),
        "merged_rows": total_merged,
        "valid_rows": valid_rows,
        "coverage_vs_gold": round(valid_rows / max(len(df_gold), 1), 4),
        "mae": round(mean_absolute_error(df["gold_rating"], df["llm_rating"]), 4),
        "accuracy": round(accuracy_score(df["gold_rating"], df["llm_rating"]), 4),
        "macro_f1": round(
            f1_score(df["gold_rating"], df["llm_rating"], labels=labels, average="macro", zero_division=0),
            4,
        ),
    }

    return metrics, df


def evaluate_all_models(
    gold_file="gold_standard_iskon.jsonl",
    output_csv="iskon_model_evaluation_summary.csv",
):
    base = Path.cwd()
    gold_path = (base / gold_file).resolve()

    if not gold_path.exists():
        raise FileNotFoundError(f"Gold file not found: {gold_path}")

    pred_files = sorted(
        p for p in base.glob("*.jsonl")
        if p.name != gold_path.name and not p.name.startswith("gold_standard")
    )

    if not pred_files:
        raise ValueError("No prediction JSONL files found for evaluation.")

    rows = []
    for pred_file in pred_files:
        metrics, _ = evaluate_one_pair(gold_path, pred_file.resolve())
        if metrics is not None:
            rows.append(metrics)

    if not rows:
        raise ValueError("No valid model evaluations were produced.")

    summary_df = pd.DataFrame(rows).sort_values(
        ["macro_f1", "accuracy", "mae"], ascending=[False, False, True]
    ).reset_index(drop=True)

    summary_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_csv}")

    return summary_df


summary_df = evaluate_all_models(
    gold_file="gold_standard_iskon.jsonl",
    output_csv="iskon_model_evaluation_summary.csv",
)

print(summary_df.to_string(index=False))
summary_df

Saved: iskon_model_evaluation_summary.csv
                  dataset  model                 gold_file                     pred_file  gold_rows  pred_rows_success  merged_rows  valid_rows  coverage_vs_gold    mae  accuracy  macro_f1
apolitical_relgious_iskon    gpt gold_standard_iskon.jsonl             gpt4o_iskon.jsonl          8                  8            8           8             1.000 0.2500     0.750    0.3000
apolitical_relgious_iskon gemini gold_standard_iskon.jsonl gemini3.1_pro_ext_iskon.jsonl          8                  8            8           8             1.000 0.2500     0.750    0.1714
apolitical_relgious_iskon claude gold_standard_iskon.jsonl   claude_sonnet_5_iskon.jsonl          8                  8            8           8             1.000 0.3750     0.625    0.1538
apolitical_relgious_iskon   qwen gold_standard_iskon.jsonl        qwen2.5_7b_iskon.jsonl          8                  7            7           7             0.875 1.7143     0.000    0.0000


,dataset,model,gold_file,pred_file,gold_rows,pred_rows_success,merged_rows,valid_rows,coverage_vs_gold,mae,accuracy,macro_f1
0,apolitical_relgious_iskon,gpt,gold_standard_iskon.jsonl,gpt4o_iskon.jsonl,8,8,8,8,1.000,0.2500,0.750,0.3000
1,apolitical_relgious_iskon,gemini,gold_standard_iskon.jsonl,gemini3.1_pro_ext_iskon.jsonl,8,8,8,8,1.000,0.2500,0.750,0.1714
2,apolitical_relgious_iskon,claude,gold_standard_iskon.jsonl,claude_sonnet_5_iskon.jsonl,8,8,8,8,1.000,0.3750,0.625,0.1538
3,apolitical_relgious_iskon,qwen,gold_standard_iskon.jsonl,qwen2.5_7b_iskon.jsonl,8,7,7,7,0.875,1.7143,0.000,0.0000
